# 02 · Build the occupation and industry panels

Outputs, all in `data/final/`:

- `occ_panel.csv`: one row per 6-digit SOC occupation × OEWS year, with employment, wage percentiles, top-code flags, exposure, Job Zone and the skill indices.
- `occ_cross.csv`: one row per occupation (2022 values), used for H1.
- `industry_panel.csv`: one row per BEA industry × year, with labour share, IP intensity and (if available) the AI-use rate.

**Skill indices** (O*NET importance scores, z-scored across occupations, then averaged):

| Index | O*NET elements | Grounding |
|---|---|---|
| Social skills | Social Perceptiveness, Coordination, Persuasion, Negotiation | Deming (2017), as discussed in *Machine, Platform, Crowd* Ch. 13 |
| Synchronization (leadership) | Coordinating the Work and Activities of Others; Guiding, Directing, and Motivating Subordinates; Developing and Building Teams; Resolving Conflicts and Negotiating with Others | This paper's "synchronization capability" |
| Ideation | Thinking Creatively; Complex Problem Solving; Making Decisions and Solving Problems | Levy & Murnane, as discussed in *The Second Machine Age* Ch. 12 |
| Protective | Mean of Social and Synchronization | H3 main measure |

In [1]:
from rwm_common import *
import difflib
oews = pd.read_csv(DATA_INT / "oews_national_long.csv", dtype={"occ_code": str})
oews["soc"] = oews["occ_code"].str.strip()
print(oews.groupby("year")["soc"].nunique())

year
2019    789
2020    789
2021    831
2022    830
2023    831
2024    831
2025    830
Name: soc, dtype: int64


### Consistent top-coding

Through May 2024, OEWS replaced annual wages at or above $239,200 with "#". The May 2025 file reports these values instead (e.g., chief executives' 90th percentile is $507,730). To keep the wage series comparable across years, the earlier top code is applied to every year in which BLS published no top-coded cells.

In [2]:
TOPCODE_USD = 239_200
WAGE_COLS = ["a_mean", "a_pct10", "a_pct25", "a_median", "a_pct75", "a_pct90"]
no_tc_years = [y for y, g in oews.groupby("year") if not g[[c + "_topcoded" for c in WAGE_COLS]].to_numpy().any()]
print("Years without BLS top-coding:", no_tc_years)
late = oews["year"].isin(no_tc_years)
for c in WAGE_COLS:
    hit = late & (oews[c] >= TOPCODE_USD)
    oews.loc[hit, c] = np.nan
    oews.loc[hit, c + "_topcoded"] = True
print(oews.groupby("year")[["a_median_topcoded", "a_pct90_topcoded"]].sum())

Years without BLS top-coding: [2025]
      a_median_topcoded  a_pct90_topcoded
year                                     
2019                  7                33
2020                  9                38
2021                 18                51
2022                 11                39
2023                 15                41
2024                 17                45
2025                 19                47


## 1. Exposure, aggregated from O*NET-SOC to 6-digit SOC

In [3]:
expo = pd.read_csv(DATA_RAW / "exposure" / "occ_level.csv")
expo["soc"] = expo["O*NET-SOC Code"].map(soc6)
ren = {"dv_rating_alpha": "alpha_gpt", "dv_rating_beta": "beta_gpt", "dv_rating_gamma": "gamma_gpt",
       "human_rating_alpha": "alpha_human", "human_rating_beta": "beta_human", "human_rating_gamma": "gamma_human"}
expo_soc = expo.rename(columns=ren).groupby("soc")[list(ren.values())].mean()
expo_soc["e2_gpt"] = expo_soc["gamma_gpt"] - expo_soc["alpha_gpt"]       # tool-dependent exposure
expo_soc["e2_human"] = expo_soc["gamma_human"] - expo_soc["alpha_human"]
print(len(expo_soc), "SOC occupations with exposure")
print("Spearman GPT vs human beta:", round(expo_soc["beta_gpt"].corr(expo_soc["beta_human"], method="spearman"), 3))

798 SOC occupations with exposure
Spearman GPT vs human beta: 0.908


## 2. O*NET Job Zones and skill indices

In [4]:
jz = pd.read_csv(DATA_INT / "onet_job_zones.csv", dtype=str)
jz["soc"] = jz["O*NET-SOC Code"].map(soc6)
jz["job_zone"] = to_num(jz["Job Zone"])
jz_soc = jz.groupby("soc")["job_zone"].mean().to_frame()
jz_soc["job_zone_cat"] = jz_soc["job_zone"].round().astype("Int64")

def onet_importance(fname: str, elements: list[str]) -> pd.DataFrame:
    d = pd.read_csv(DATA_INT / fname, dtype=str)
    d = d[d["Scale ID"] == "IM"].copy()
    d["value"] = to_num(d["Data Value"])
    d["soc"] = d["O*NET-SOC Code"].map(soc6)
    names = d["Element Name"].str.strip()
    out = {}
    for el in elements:
        hit = names.str.lower() == el.lower()
        if not hit.any():                                   # tolerate small renamings
            close = difflib.get_close_matches(el, names.unique().tolist(), n=1, cutoff=0.8)
            if not close:
                raise KeyError(f"O*NET element not found: {el}")
            print(f"  '{el}' matched to '{close[0]}'")
            hit = names == close[0]
        out[el] = d[hit].groupby("soc")["value"].mean()
    return pd.DataFrame(out)

SOCIAL = ["Social Perceptiveness", "Coordination", "Persuasion", "Negotiation"]
SYNC = ["Coordinating the Work and Activities of Others",
        "Guiding, Directing, and Motivating Subordinates",
        "Developing and Building Teams",
        "Resolving Conflicts and Negotiating with Others"]
IDEATION_SK = ["Complex Problem Solving"]
IDEATION_WA = ["Thinking Creatively", "Making Decisions and Solving Problems"]

sk = onet_importance("onet_skills.csv", SOCIAL + IDEATION_SK)
wa = onet_importance("onet_work_activities.csv", SYNC + IDEATION_WA)
items = sk.join(wa, how="outer")

def index_of(cols):
    return pd.concat([zscore(items[c]) for c in cols], axis=1).mean(axis=1)

idx = pd.DataFrame({
    "social_idx": index_of(SOCIAL),
    "sync_idx": index_of(SYNC),
    "ideation_idx": index_of(IDEATION_SK + IDEATION_WA),
})
idx["protective_idx"] = idx[["social_idx", "sync_idx"]].mean(axis=1)
for c in idx:                                   # re-standardise the composites
    idx[c] = zscore(idx[c])
print(idx.describe().round(2))
rel = pd.Series({"Social skills": cronbach_alpha(items[SOCIAL]),
                 "Synchronization": cronbach_alpha(items[SYNC]),
                 "Ideation": cronbach_alpha(items[IDEATION_SK + IDEATION_WA])}, name="cronbach_alpha")
save_table(rel.to_frame(), "table_s0_index_reliability", "Internal consistency of the O*NET skill indices")
print(rel.round(3))

       social_idx  sync_idx  ideation_idx  protective_idx
count      774.00    774.00        774.00          774.00
mean         0.00      0.00         -0.00            0.00
std          1.00      1.00          1.00            1.00
min         -2.46     -2.95         -2.95           -2.78
25%         -0.76     -0.70         -0.61           -0.72
50%         -0.10     -0.02          0.05           -0.10
75%          0.67      0.66          0.81            0.68
max          2.64      2.69          2.17            2.89
Saved table table_s0_index_reliability
Social skills      0.925
Synchronization    0.899
Ideation           0.798
Name: cronbach_alpha, dtype: float64


## 3. Occupation panel

Codes ending in `0` in OEWS are aggregates that O*NET does not rate; they drop out at the merge. The match rate is printed.

In [5]:
panel = (oews.merge(expo_soc, left_on="soc", right_index=True, how="left")
             .merge(jz_soc, left_on="soc", right_index=True, how="left")
             .merge(idx, left_on="soc", right_index=True, how="left"))
for y, g in panel.groupby("year"):
    m = g["beta_gpt"].notna()
    print(f"{y}: {m.mean():.1%} of occupations, {g.loc[m,'tot_emp'].sum()/g['tot_emp'].sum():.1%} of employment matched to exposure")

# A SOC whose 90th percentile is top-coded in any year is flagged for the P90 models.
tc = panel.groupby("soc")["a_pct90_topcoded"].any().rename("p90_topcoded_any")
panel = panel.merge(tc, left_on="soc", right_index=True, how="left")
panel.to_csv(DATA_FIN / "occ_panel.csv", index=False)

cross = panel[panel["year"] == BASE_YEAR_WEIGHTS].dropna(subset=["beta_gpt"]).copy()
cross["lmed"] = np.log(cross["a_median"])
cross.to_csv(DATA_FIN / "occ_cross.csv", index=False)
print("cross-section:", len(cross), "occupations;", f"{cross['tot_emp'].sum()/1e6:.1f}m workers")

2019: 89.9% of occupations, 88.4% of employment matched to exposure
2020: 89.9% of occupations, 87.8% of employment matched to exposure
2021: 92.9% of occupations, 92.7% of employment matched to exposure
2022: 92.9% of occupations, 92.8% of employment matched to exposure
2023: 92.9% of occupations, 92.8% of employment matched to exposure
2024: 92.9% of occupations, 92.6% of employment matched to exposure
2025: 92.9% of occupations, 92.5% of employment matched to exposure
cross-section: 771 occupations; 137.2m workers


## 4. Industry panel (BEA)

Labour share = compensation of employees ÷ value added. Only the most detailed industries are kept, so nothing is double-counted; the check below confirms that they sum to roughly total GDP.

In [6]:
raw = pd.read_csv(DATA_INT / "bea_components_raw.csv", dtype=str)
col = {c.lower(): c for c in raw.columns}
ind_c, desc_c = col["industry"], col["industrydescription"]
raw["year"] = to_num(raw[col["year"]]).astype(int)
# GDP-by-Industry values are in billions of dollars; the API usually omits UNIT_MULT.
mult = to_num(raw[col["unit_mult"]]).fillna(9) if "unit_mult" in col else 9
raw["value"] = to_num(raw[col["datavalue"]]) * (10.0 ** mult)

COMP = {"compensation of employees": "comp", "gross operating surplus": "gos",
        "taxes on production and imports less subsidies": "tax"}
raw["component"] = raw[desc_c].str.strip().str.lower().map(COMP).fillna("va")
if not (raw["component"] == "comp").any():
    print(raw[[ind_c, desc_c]].drop_duplicates().head(30))
    raise RuntimeError("Could not find compensation rows; the BEA layout differs from what this code expects. "
                       "Send me the printout above.")

names = raw[raw["component"] == "va"].groupby(ind_c)[desc_c].first()
wide = raw.pivot_table(index=[ind_c, "year"], columns="component", values="value", aggfunc="first").reset_index()
wide = wide.rename(columns={ind_c: "ind"})
wide["name"] = wide["ind"].map(names)
wide["va_sum"] = wide[["comp", "gos", "tax"]].sum(axis=1, min_count=3)
wide["va"] = wide["va"].fillna(wide["va_sum"])
print("Max gap between reported VA and comp+GOS+tax:",
      f"{(wide['va'] - wide['va_sum']).abs().div(wide['va']).max():.2%}")

# Keep leaf industries: drop known aggregates and any code that prefixes a longer code.
codes = wide["ind"].astype(str).unique().tolist()
AGG = {"GDP", "PVT", "PGOOD", "PSERV", "ICT", "ICTPRO", "11", "21", "31G", "33DG", "31ND", "44RT", "48TW",
       "51", "FIRE", "52", "53", "531", "PROF", "54", "56", "6", "62", "7", "71", "72", "G", "GF", "GFG", "GSL"}
# 531 (Real estate) = HS (Housing) + ORE (Other real estate); the prefix rule cannot detect that.
is_parent = {c for c in codes if any(o != c and o.startswith(c) for o in codes)}
gdp_code = next((c for c in codes if c.upper() == "GDP"), None)
if gdp_code is None:
    gdp_code = names[names.str.contains("gross domestic product", case=False)].index[0]
leaf = [c for c in codes if c not in AGG and c not in is_parent]
gdp = wide[wide["ind"] == gdp_code].set_index("year")["va"]
cover = wide[wide["ind"].isin(leaf)].groupby("year")["va"].sum() / gdp
print(f"{len(leaf)} leaf industries; their value added sums to {cover.min():.1%}-{cover.max():.1%} of GDP")
if not (0.95 < cover.median() < 1.05):
    print("WARNING: leaf industries do not add up to GDP. Inspect `leaf` and adjust AGG.")
# H5 covers private industries: government has no private IP investment and no operating surplus to speak of.
private = [c for c in leaf if not str(c).startswith("G")]
if "PVT" in codes:
    pvt = wide[wide["ind"] == "PVT"].set_index("year")["va"]
    pc = wide[wide["ind"].isin(private)].groupby("year")["va"].sum() / pvt
    print(f"{len(private)} private leaf industries; {pc.min():.1%}-{pc.max():.1%} of private-industry value added")
leaf = private

ind = wide[wide["ind"].isin(leaf)].copy()
ind["labor_share"] = ind["comp"] / ind["va"]
ind = ind[ind["va"] > 0]
# Very small industries have erratic labour shares (e.g., funds and trusts, where compensation can exceed value added).
va_share = ind.groupby("ind")["va"].sum() / ind["va"].sum()
tiny = va_share[va_share < 0.001].index
print("Dropped as < 0.1% of private value added:", {c: names.get(c) for c in tiny})
ind = ind[~ind["ind"].isin(tiny)]
print(ind.groupby("year")["labor_share"].describe()[["mean", "50%"]].T.round(3))

Max gap between reported VA and comp+GOS+tax: 10.00%
71 leaf industries; their value added sums to 100.0%-100.0% of GDP
66 private leaf industries; 100.0%-100.0% of private-industry value added
Dropped as < 0.1% of private value added: {'315AL': 'Apparel and leather and allied products', '483': 'Water transportation', '525': 'Funds, trusts, and other financial vehicles'}
year   1997   1998   1999   2000   2001   2002   2003   2004   2005   2006  \
mean  0.581  0.589  0.596  0.614  0.604  0.589  0.581  0.571  0.562  0.555   
50%   0.582  0.603  0.611  0.616  0.634  0.620  0.617  0.596  0.592  0.583   

year  ...   2016   2017   2018   2019   2020   2021   2022   2023   2024  \
mean  ...  0.541  0.542  0.543  0.546  0.567  0.545  0.533  0.530  0.533   
50%   ...  0.562  0.544  0.535  0.560  0.582  0.553  0.513  0.546  0.550   

year   2025  
mean  0.533  
50%   0.546  

[2 rows x 29 columns]


### IP intensity

The Fixed Assets tables name industries slightly differently from GDP by Industry. Names are matched exactly first, then by close string match; the mapping is written to `data/interim/ip_industry_mapping.csv` for checking.

In [7]:
ipr = pd.read_csv(DATA_INT / "bea_ip_investment_raw.csv", dtype=str)
c2 = {c.lower(): c for c in ipr.columns}
ipr["year"] = to_num(ipr[c2["timeperiod"]]).astype(int)
mult = to_num(ipr[c2["unit_mult"]]).fillna(9) if "unit_mult" in c2 else 9   # Fixed Assets: billions
ipr["ip_inv"] = to_num(ipr[c2["datavalue"]]) * (10.0 ** mult)
ipr["fa_name"] = ipr[c2["linedescription"]].str.strip()

def norm(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    s = re.sub(r"\b(and|industries|industry|except|of|the|other)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# Hand-checked matches for GDP-by-Industry names that differ from Fixed Assets line names.
# Each value is a list of Fixed Assets lines whose investment is SUMMED for that industry.
# Checked against the October 2026 BEA tables; edit if BEA renames lines.
MANUAL_FA_MATCH = {
    "Rail transportation": ["Railroad transportation"],
    "Publishing industries, except internet (includes software)": ["Publishing industries (includes software)"],
    "Data processing, internet publishing, and other information services": ["Information and data processing services"],
    "Federal Reserve banks, credit intermediation, and related activities": ["Federal Reserve banks",
                                                                           "Credit intermediation and related activities"],
    "Other real estate": ["Real estate"],
}
fa_names = set(ipr["fa_name"].drop_duplicates())
fa_norm = {norm(n): n for n in fa_names}
rows = []
for code_, nm in ind.drop_duplicates("ind")[["ind", "name"]].itertuples(index=False):
    nm = str(nm).strip()
    manual = [m for m in MANUAL_FA_MATCH.get(nm, []) if m in fa_names]
    if manual:
        rows += [(code_, nm, m, "manual") for m in manual]
    elif norm(nm) in fa_norm:
        rows.append((code_, nm, fa_norm[norm(nm)], "exact"))
    else:
        rows.append((code_, nm, None, "none"))
mapping = pd.DataFrame(rows, columns=["ind", "bea_name", "fa_name", "match"])
mapping.to_csv(DATA_INT / "ip_industry_mapping.csv", index=False)
print(mapping.drop_duplicates("ind")["match"].value_counts())
print("Unmatched industries (no IP intensity; Housing is expected here):")
print(mapping.loc[mapping["match"] == "none", ["ind", "bea_name"]].to_string(index=False))
dup = mapping.dropna(subset=["fa_name"]).groupby("fa_name")["ind"].nunique()
if (dup > 1).any():
    print("WARNING: a Fixed Assets line is mapped to more than one industry:", dup[dup > 1].to_dict())

ipm = ipr.merge(mapping[["ind", "fa_name"]].dropna(), on="fa_name")[["ind", "year", "ip_inv"]]
ipm = ipm.groupby(["ind", "year"], as_index=False)["ip_inv"].sum()
ind = ind.merge(ipm, on=["ind", "year"], how="left")
ind["ip_share_va"] = ind["ip_inv"] / ind["va"]
pre = ind[(ind["year"] >= 1997) & (ind["year"] <= 2007)].groupby("ind")["ip_share_va"].mean()
ind["ip_int_pre"] = ind["ind"].map(pre)
print(f"IP intensity available for {pre.notna().sum()} of {ind['ind'].nunique()} industries")
if not (0 < pre.median() < 1):
    print("WARNING: IP intensity outside 0-1; check the units of the two BEA tables.")
print(pre.sort_values(ascending=False).head(10).rename(names).round(3))

match
exact     57
manual     5
none       1
Name: count, dtype: int64
Unmatched industries (no IP intensity; Housing is expected here):
ind bea_name
 HS  Housing
IP intensity available for 62 of 63 industries
ind
Motion picture and sound recording industries                           0.484
Computer and electronic products                                        0.258
Data processing, internet publishing, and other information services    0.213
Chemical products                                                       0.206
Publishing industries, except internet (includes software)              0.184
Other transportation equipment                                          0.149
Motor vehicles, bodies and trailers, and parts                          0.136
Performing arts, spectator sports, museums, and related activities      0.134
Computer systems design and related services                            0.107
Broadcasting and telecommunications                                     0.103
Name: 

### AI use by sector (Census BTOS, optional)

The parser looks for the AI-use question and the "Yes" answer in the sector file's response-estimates sheet. If the layout differs, it prints the sheet structure and continues without this variable.

In [8]:
def two_digit_sector(code):
    # BEA industry code -> BTOS sector code (BTOS uses 31, 44 and 48 for the combined sectors)
    c = str(code)
    if c in ("HS", "ORE") or c.startswith("53"): return "53"
    if c.startswith("G"): return None
    if c[:1] == "3": return "31"
    if c[:2] in ("44", "45") or c == "4A0": return "44"
    if c[:2] in ("48", "49"): return "48"
    return c[:2]

ind["sector2"] = ind["ind"].map(two_digit_sector)
ai = None
f = DATA_RAW / "btos" / "sector.xlsx"
if f.exists():
    try:
        xl = pd.ExcelFile(f)
        sheet = next(s for s in xl.sheet_names if "response" in s.lower())
        bt = xl.parse(sheet, dtype=str)
        bt.columns = [str(c).strip() for c in bt.columns]
        qcol = next(c for c in bt.columns if "question" in c.lower() and "id" not in c.lower())
        acol = next(c for c in bt.columns if c.lower().startswith("answer") and "id" not in c.lower())
        scol = next(c for c in bt.columns if "sector" in c.lower())
        pcols = [c for c in bt.columns if re.fullmatch(r"\d{6}", c)]
        # Current use ("in the last two weeks"), not expected use in the next six months.
        # Census renamed AI "Super Intelligence (SI)" in later files and widened the question to
        # "any business function" in November 2025; the sector file carries values only for the
        # waves asked under the current wording, so all waves with numbers are averaged.
        m = (bt[qcol].str.contains(r"last two weeks", case=False, na=False)
             & bt[qcol].str.contains(r"intelligence|\bAI\b|\bSI\b", case=False, regex=True, na=False)
             & bt[acol].str.strip().str.lower().eq("yes"))
        print("AI question used:", bt.loc[m, qcol].iloc[0][:160], "...")
        long = bt[m].melt(id_vars=[scol], value_vars=pcols, var_name="period", value_name="share")
        long["share"] = to_num(long["share"].astype(str).str.replace("%", "", regex=False))
        long = long.dropna(subset=["share"])
        print(f"AI-use waves used: {long['period'].min()} to {long['period'].max()} ({long['period'].nunique()} waves)")
        ai = long.groupby(scol)["share"].mean().rename("ai_use_pct")
        ai.index = ai.index.astype(str).str.strip().str[:2]
        ai = ai[ai.index.str.isdigit()]
        print(ai.round(1))
    except Exception as e:
        print("BTOS sector file not parsed:", e)
        try:
            print(xl.sheet_names); print(bt.head())
        except Exception:
            pass
else:
    print("No BTOS sector file; AI-use check will be skipped.")
ind["ai_use_pct"] = ind["sector2"].map(ai) if ai is not None else np.nan
ind.to_csv(DATA_FIN / "industry_panel.csv", index=False)
print("industry panel:", ind.shape)

C:\Users\91838\AppData\Local\Temp\ipykernel_6108\2375006543.py:18: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  bt = xl.parse(sheet, dtype=str)


AI question used: In the last two weeks, did this business use Super Intelligence or SI (formerly referred to as Artificial Intelligence or AI) in any of its business functions?  ...
AI-use waves used: 202524 to 202621 (24 waves)
Sector
11     7.6
21     9.0
22    14.5
23    11.9
31    16.8
42    16.9
44    14.6
48     9.0
51    41.2
52    34.1
53    26.2
54    37.7
55    30.9
56    18.7
61    32.5
62    22.3
71    18.5
72     8.6
81    11.0
Name: ai_use_pct, dtype: float64
industry panel: (1827, 14)


## Check

In [9]:
p = pd.read_csv(DATA_FIN / "occ_panel.csv", dtype={"soc": str})
print("occupation-years:", len(p), "| occupations:", p["soc"].nunique(), "| years:", sorted(int(y) for y in p["year"].unique()))
print("with exposure:", p.dropna(subset=["beta_gpt"])["soc"].nunique(),
      "| with protective index:", p.dropna(subset=["protective_idx"])["soc"].nunique())
i = pd.read_csv(DATA_FIN / "industry_panel.csv")
print("industries:", i["ind"].nunique(), "| years:", i["year"].min(), "-", i["year"].max(),
      "| with IP intensity:", i.dropna(subset=["ip_int_pre"])["ind"].nunique())

occupation-years: 5731 | occupations: 861 | years: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
with exposure: 772 | with protective index: 750
industries: 63 | years: 1997 - 2025 | with IP intensity: 62
